# Design Hub v2: Control vs Perturbation Experiment

This notebook walks you through a simple experiment comparing a **control** CTSM simulation against a **perturbation** case where we modify precipitation forcing.

**Workflow:**
1. Configure site and year
2. Run the control case (no modifications)
3. Run the perturbation case (scale precipitation by +20%)
4. Load and compare simulation outputs
5. Visualize the difference

**Prerequisites:** This notebook runs inside the ExSoil container where `run_neon_v2.py` and CTSM are available.

## 1. Configuration

Set your NEON site and simulation year here. Everything else flows from these two values.

In [ ]:
# ── Configuration ────────────────────────────────────────────
NEON_SITE = "ABBY"          # 4-letter NEON site code
YEAR = "2018"               # Simulation year
OUTPUT_ROOT = "~/CLM-NEON"  # Where simulation outputs go

# Perturbation settings
TRANSFORM_VAR = "PRECTmms"  # Variable to perturb (precipitation)
TRANSFORM_METHOD = "scaling" # scaling | add | seasonal_scaling | noise
TRANSFORM_VALUE = 1.2        # 1.2 = +20% precipitation

# Pass site to shell commands
import os
os.environ["site"] = NEON_SITE
os.environ["output_root"] = OUTPUT_ROOT

print(f"Site: {NEON_SITE}")
print(f"Year: {YEAR}")
print(f"Perturbation: {TRANSFORM_VAR} x {TRANSFORM_VALUE}")

## 2. Run the Control Case

This runs a standard CTSM transient simulation with unmodified forcing data. This may take several minutes.

In [ ]:
%%bash
# Run control case (no perturbation)
run_neon --neon-sites $site --output-root $output_root --overwrite

## 3. Run the Perturbation Case

This runs the same simulation but with precipitation scaled by +20%. The `--transform-tag` creates a separate output directory so it doesn't overwrite the control.

In [ ]:
%%bash
# Run perturbation case: +20% precipitation
run_neon_v2 --neon-sites $site --output-root $output_root --overwrite \
    --transform-var PRECTmms \
    --transform-method scaling \
    --transform-value 1.2 \
    --transform-tag precip_plus20

## 4. Load Simulation Outputs

Load both control and perturbation results for comparison. We focus on **H2OSOI** (soil moisture) to see how precipitation changes affect soil water content.

In [ ]:
%matplotlib inline

import time
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from glob import glob
from os.path import join, expanduser

# Variable to compare
VAR = "H2OSOI"

# ── Load control case ────────────────────────────────────────
control_path = expanduser(f"{OUTPUT_ROOT}/{NEON_SITE}.transient/lnd/hist/")
control_files = sorted(glob(join(control_path, f"{NEON_SITE}.transient.clm2.h1.{YEAR}*.nc")))
print(f"Control files: {len(control_files)}")

# ── Load perturbation case ───────────────────────────────────
perturb_path = expanduser(f"{OUTPUT_ROOT}/{NEON_SITE}.transient_precip_plus20/lnd/hist/")
perturb_files = sorted(glob(join(perturb_path, f"{NEON_SITE}.transient*.clm2.h1.{YEAR}*.nc")))

# If the perturbation path doesn't match, try alternative naming
if not perturb_files:
    # Check what directories exist
    import os
    clm_dir = expanduser(OUTPUT_ROOT)
    dirs = [d for d in os.listdir(clm_dir) if NEON_SITE in d]
    print(f"Available case directories: {dirs}")
    print("Adjust perturb_path above to match your perturbation case directory.")
else:
    print(f"Perturbation files: {len(perturb_files)}")

In [ ]:
# ── Read both datasets ────────────────────────────────────────
from analytics_modules import ctsm_sim_depth

print("Loading control...")
df_control = ctsm_sim_depth(control_files, VAR, levsoi=2.5)
df_control = df_control.rename(columns={VAR: f"{VAR}_control"})
print(f"  {len(df_control)} rows")

print("Loading perturbation...")
df_perturb = ctsm_sim_depth(perturb_files, VAR, levsoi=2.5)
df_perturb = df_perturb.rename(columns={VAR: f"{VAR}_perturb"})
print(f"  {len(df_perturb)} rows")

In [ ]:
# ── Merge on time ─────────────────────────────────────────────
df = pd.merge(
    df_control[["time", f"{VAR}_control"]],
    df_perturb[["time", f"{VAR}_perturb"]],
    on="time",
    how="inner",
)
df["difference"] = df[f"{VAR}_perturb"] - df[f"{VAR}_control"]

# Aggregate to daily
df["date"] = df["time"].dt.date
df_daily = df.groupby("date").mean(numeric_only=True).reset_index()
df_daily["date"] = pd.to_datetime(df_daily["date"])

print(f"Merged: {len(df)} half-hourly, {len(df_daily)} daily rows")
df_daily.head()

## 5. Visualization

### 5a. Time series: Control vs Perturbation

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# ── Top panel: both time series ──────────────────────────────
ax = axes[0]
ax.plot(df_daily["date"], df_daily[f"{VAR}_control"], label="Control", color="steelblue")
ax.plot(df_daily["date"], df_daily[f"{VAR}_perturb"], label=f"Perturbation ({TRANSFORM_VAR} x{TRANSFORM_VALUE})", color="coral")
ax.set_ylabel(f"{VAR} [mm3/mm3]", fontsize=12)
ax.set_title(f"{NEON_SITE} - Soil Moisture at 2.5m: Control vs +20% Precipitation ({YEAR})", fontsize=14, fontweight="bold")
ax.legend(fontsize=11)
ax.grid(alpha=0.3)

# ── Bottom panel: difference ─────────────────────────────────
ax = axes[1]
ax.fill_between(df_daily["date"], df_daily["difference"], alpha=0.4, color="green", label="Perturb - Control")
ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_ylabel("Difference [mm3/mm3]", fontsize=12)
ax.set_xlabel("Time", fontsize=12)
ax.set_title("Soil Moisture Difference (Perturbation - Control)", fontsize=13)
ax.legend(fontsize=11)
ax.grid(alpha=0.3)

plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

# Summary stats
mean_diff = df_daily["difference"].mean()
max_diff = df_daily["difference"].max()
print(f"Mean daily difference: {mean_diff:.6f} mm3/mm3")
print(f"Max daily difference:  {max_diff:.6f} mm3/mm3")

### 5b. Soil Profile Visualization

Compare the full soil depth profile over time for both cases.

In [ ]:
from analytics_modules import plot_soil_profile_timeseries

# Control case - soil moisture profile
print("Control case - Soil Moisture Profile:")
ds_control = plot_soil_profile_timeseries(NEON_SITE, "H2OSOI", year=int(YEAR))